In [ ]:
import pandas as pd
import numpy as np
import math
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import os
from google.colab import drive

drive.mount('/content/drive')

# Load main DBD dataset
!cp "/content/drive/MyDrive/hasil_data_gabungan_DBD_iklim lokal.csv" "./data.csv"
df = pd.read_csv("data.csv")

# Clean numeric columns
for col in ["Kasus DBD", "Curah Hujan", "Kelembapan", "Suhu"]:
    df[col] = df[col].astype(str).str.replace(",", "").str.strip()
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.dropna(inplace=True)

# Load DMI & ONI data
!cp "/content/drive/MyDrive/DATA/ONI (Tahun).xlsx" "./ONI_3.xlsx"
!cp "/content/drive/MyDrive/DATA/DMI pertahun (Tahun).xlsx" "./DMI_pertahun.xlsx"
dmi_df = pd.read_excel("DMI_pertahun.xlsx")
oni_df = pd.read_excel("ONI_3.xlsx")

# Merge DMI and ONI into main dataframe
df = df.merge(dmi_df, on="Tahun", how="left")
df = df.merge(oni_df, on="Tahun", how="left")
df.dropna(inplace=True)

# Sequence preparation
def create_sequences(data, n_steps=2):
    X, y = [], []
    for i in range(len(data) - n_steps):
        X.append(data[i:i+n_steps, 1:])  # input: all but Kasus
        y.append(data[i+n_steps, 0])     # target: Kasus DBD
    return np.array(X), np.array(y)

def inverse_scale(preds, scaler):
    dummy = np.zeros((len(preds), 5))  # 5 dummy features to pad to 6
    return scaler.inverse_transform(np.hstack((preds, dummy)))[:, 0]

# Config
n_steps = 2
performance_results = []
provinces = df["Nama Provinsi"].unique()

training_histories = {}

for province in provinces:
    print(f"\nProcessing: {province}")
    df_prov = df[df["Nama Provinsi"] == province].sort_values("Tahun").copy()

    if len(df_prov) < 5:
        print(f"Skipping {province} due to insufficient data.")
        continue

    # Normalize
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(df_prov[["Kasus DBD", "Curah Hujan", "Kelembapan", "Suhu", "DMI", "ONI"]])
    X, y = create_sequences(scaled, n_steps)
    if len(X) < 1:
        print(f"Skipping {province} due to insufficient sequences.")
        continue

    # Split
    split = int(len(X) * 0.8)
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    # 1-Layer LSTM
    model = Sequential()
    model.add(LSTM(54, input_shape=(n_steps, X.shape[2])))
    model.add(Dropout(0.2))
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

    # Train
    early_stop = EarlyStopping(monitor='val_loss', patience=200, restore_best_weights=True)
    history = model.fit(X_train, y_train, epochs=4000, batch_size=1, verbose=0,
              callbacks=[early_stop], validation_data=(X_test, y_test))
    training_histories[province] = history.history

    # Predict
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    y_train_inv = inverse_scale(y_train.reshape(-1, 1), scaler)
    y_train_pred_inv = inverse_scale(y_train_pred, scaler)
    y_test_inv = inverse_scale(y_test.reshape(-1, 1), scaler)
    y_test_pred_inv = inverse_scale(y_test_pred, scaler)

    # Metrics
    train_rmse = math.sqrt(mean_squared_error(y_train_inv, y_train_pred_inv))
    train_mae = mean_absolute_error(y_train_inv, y_train_pred_inv)
    test_rmse = math.sqrt(mean_squared_error(y_test_inv, y_test_pred_inv))
    test_mae = mean_absolute_error(y_test_inv, y_test_pred_inv)
    train_r2 = r2_score(y_train_inv, y_train_pred_inv)
    test_r2 = r2_score(y_test_inv, y_test_pred_inv)

    performance_results.append({
        'Province': province,
        'Model': 'LSTM_1Layer_6Features',
        'Train RMSE': train_rmse,
        'Train MAE': train_mae,
        'Train R2': train_r2,
        'Test RMSE': test_rmse,
        'Test MAE': test_mae,
        'Test R2': test_r2
    })

# Save results
performance_df = pd.DataFrame(performance_results)
performance_df.to_csv("performance_results_1Layer.csv", index=False)
print("\nPerformance Summary:")
print(performance_df)

Mounted at /content/drive

Processing: ACEH


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 233ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms/step

Processing: SUMATERA UTARA


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 190ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 181ms/step

Processing: SUMATERA BARAT


...

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 169ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step

Processing: PAPUA BARAT


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 255ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 265ms/step

Processing: PAPUA


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 180ms/step

Performance Summary:
                Province                  Model   Train RMSE    Train MAE  \
0                   ACEH  LSTM_1Layer_6Features   825.572446   734.642408   
1         SUMATERA UTARA  LSTM_1Layer_6Features  1249.645433  1065.715178   
2         SUMATERA BARAT  LSTM_1Layer_6Features   286.831511   210.346663   
3                   RIAU  LSTM_1Layer_6Features  1306.338552  1067.545697   
4                  JAMBI  LSTM_1Layer_6Features   646.595980   549.441832   
5       SUMATERA SELATAN  LSTM_1Layer_6Features   163.434980   137.881053   
6               BENGKULU  LSTM_1Layer_6Features   164.210567   112.757337   
7                LAMPUNG  LSTM_1Layer_6Features   356.211767   292.531420   
8   KEP. BANGKA BELITUNG  LSTM_1Layer_6Features   148.476488   116.484945   
9              KEP. RIAU  LSTM_1Layer_6Features    80.275674    74.627643   
10           DKI JAKARTA  LSTM_1Layer_6Features  5476

In [ ]:
import pandas as pd
import numpy as np
import math
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import os
from google.colab import drive

drive.mount('/content/drive')

# Load main DBD dataset
!cp "/content/drive/MyDrive/hasil_data_gabungan_DBD_iklim lokal.csv" "./data.csv"
df = pd.read_csv("data.csv")

# Clean numeric columns
for col in ["Kasus DBD", "Curah Hujan", "Kelembapan", "Suhu"]:
    df[col] = df[col].astype(str).str.replace(",", "").str.strip()
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.dropna(inplace=True)

# Load DMI & ONI data
!cp "/content/drive/MyDrive/DATA/ONI (Tahun).xlsx" "./ONI_3.xlsx"
!cp "/content/drive/MyDrive/DATA/DMI pertahun (Tahun).xlsx" "./DMI_pertahun.xlsx"
dmi_df = pd.read_excel("DMI_pertahun.xlsx")
oni_df = pd.read_excel("ONI_3.xlsx")

# Merge DMI and ONI into main dataframe
df = df.merge(dmi_df, on="Tahun", how="left")
df = df.merge(oni_df, on="Tahun", how="left")
df.dropna(inplace=True)

# Sequence preparation
def create_sequences(data, n_steps=2):
    X, y = [], []
    for i in range(len(data) - n_steps):
        X.append(data[i:i+n_steps, 1:])  # input: all but Kasus
        y.append(data[i+n_steps, 0])     # target: Kasus DBD
    return np.array(X), np.array(y)

def inverse_scale(preds, scaler):
    dummy = np.zeros((len(preds), 5))  # 5 dummy features to pad to 6
    return scaler.inverse_transform(np.hstack((preds, dummy)))[:, 0]

# Config
n_steps = 2
performance_results = []
provinces = df["Nama Provinsi"].unique()

training_histories = {}

for province in provinces:
    print(f"\nProcessing: {province}")
    df_prov = df[df["Nama Provinsi"] == province].sort_values("Tahun").copy()

    if len(df_prov) < 5:
        print(f"Skipping {province} due to insufficient data.")
        continue

    # Normalize
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(df_prov[["Kasus DBD", "Curah Hujan", "Kelembapan", "Suhu", "DMI", "ONI"]])
    X, y = create_sequences(scaled, n_steps)
    if len(X) < 1:
        print(f"Skipping {province} due to insufficient sequences.")
        continue

    # Split
    split = int(len(X) * 0.8)
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    # 1-Layer LSTM
    model = Sequential()
    model.add(LSTM(54, input_shape=(n_steps, X.shape[2])))
    model.add(Dropout(0.2))
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=0.005), loss='mse')

    # Train
    early_stop = EarlyStopping(monitor='val_loss', patience=200, restore_best_weights=True)
    history = model.fit(X_train, y_train, epochs=4000, batch_size=1, verbose=0,
              callbacks=[early_stop], validation_data=(X_test, y_test))
    training_histories[province] = history.history

    # Predict
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    y_train_inv = inverse_scale(y_train.reshape(-1, 1), scaler)
    y_train_pred_inv = inverse_scale(y_train_pred, scaler)
    y_test_inv = inverse_scale(y_test.reshape(-1, 1), scaler)
    y_test_pred_inv = inverse_scale(y_test_pred, scaler)

    # Metrics
    train_rmse = math.sqrt(mean_squared_error(y_train_inv, y_train_pred_inv))
    train_mae = mean_absolute_error(y_train_inv, y_train_pred_inv)
    test_rmse = math.sqrt(mean_squared_error(y_test_inv, y_test_pred_inv))
    test_mae = mean_absolute_error(y_test_inv, y_test_pred_inv)
    train_r2 = r2_score(y_train_inv, y_train_pred_inv)
    test_r2 = r2_score(y_test_inv, y_test_pred_inv)

    performance_results.append({
        'Province': province,
        'Model': 'LSTM_1Layer_6Features',
        'Train RMSE': train_rmse,
        'Train MAE': train_mae,
        'Train R2': train_r2,
        'Test RMSE': test_rmse,
        'Test MAE': test_mae,
        'Test R2': test_r2
    })

# Save results
performance_df = pd.DataFrame(performance_results)
performance_df.to_csv("performance_results_1Layer.csv", index=False)
print("\nPerformance Summary:")
print(performance_df)

Mounted at /content/drive

Processing: ACEH


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 252ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step

Processing: SUMATERA UTARA


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 186ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 204ms/step

Processing: SUMATERA BARAT


...

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 184ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 195ms/step

Processing: PAPUA BARAT


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 186ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 192ms/step

Processing: PAPUA


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 181ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 203ms/step

Performance Summary:
                Province                  Model   Train RMSE    Train MAE  \
0                   ACEH  LSTM_1Layer_6Features   715.065535   620.314028   
1         SUMATERA UTARA  LSTM_1Layer_6Features   463.799471   400.466842   
2         SUMATERA BARAT  LSTM_1Layer_6Features   204.934811   141.999411   
3                   RIAU  LSTM_1Layer_6Features   371.262104   328.613842   
4                  JAMBI  LSTM_1Layer_6Features   357.374905   281.747152   
5       SUMATERA SELATAN  LSTM_1Layer_6Features   349.061370   269.140388   
6               BENGKULU  LSTM_1Layer_6Features   181.890075   137.460834   
7                LAMPUNG  LSTM_1Layer_6Features   400.779463   302.387523   
8   KEP. BANGKA BELITUNG  LSTM_1Layer_6Features   296.024663   234.640625   
9              KEP. RIAU  LSTM_1Layer_6Features   106.386620    83.225232   
10           DKI JAKARTA  LSTM_1Layer_6Features  3981

In [ ]:
import pandas as pd
import numpy as np
import math
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import os
from google.colab import drive

drive.mount('/content/drive')

# Load main DBD dataset
!cp "/content/drive/MyDrive/hasil_data_gabungan_DBD_iklim lokal.csv" "./data.csv"
df = pd.read_csv("data.csv")

# Clean numeric columns
for col in ["Kasus DBD", "Curah Hujan", "Kelembapan", "Suhu"]:
    df[col] = df[col].astype(str).str.replace(",", "").str.strip()
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.dropna(inplace=True)

# Load DMI & ONI data
!cp "/content/drive/MyDrive/DATA/ONI (Tahun).xlsx" "./ONI_3.xlsx"
!cp "/content/drive/MyDrive/DATA/DMI pertahun (Tahun).xlsx" "./DMI_pertahun.xlsx"
dmi_df = pd.read_excel("DMI_pertahun.xlsx")
oni_df = pd.read_excel("ONI_3.xlsx")

# Merge DMI and ONI into main dataframe
df = df.merge(dmi_df, on="Tahun", how="left")
df = df.merge(oni_df, on="Tahun", how="left")
df.dropna(inplace=True)

# Sequence preparation
def create_sequences(data, n_steps=2):
    X, y = [], []
    for i in range(len(data) - n_steps):
        X.append(data[i:i+n_steps, 1:])  # input: all but Kasus
        y.append(data[i+n_steps, 0])     # target: Kasus DBD
    return np.array(X), np.array(y)

def inverse_scale(preds, scaler):
    dummy = np.zeros((len(preds), 5))  # 5 dummy features to pad to 6
    return scaler.inverse_transform(np.hstack((preds, dummy)))[:, 0]

# Config
n_steps = 2
performance_results = []
provinces = df["Nama Provinsi"].unique()

training_histories = {}

for province in provinces:
  # if province == "KEP. BANGKA BELITUNG":
   if province == "NUSA TENGGARA BARAT"or province == "NUSA TENGGARA TIMUR"or province == "KEP. BANGKA BELITUNG":
    print(f"\nProcessing: {province}")
    df_prov = df[df["Nama Provinsi"] == province].sort_values("Tahun").copy()

    if len(df_prov) < 5:
        print(f"Skipping {province} due to insufficient data.")
        continue

    # Normalize
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(df_prov[["Kasus DBD", "Curah Hujan", "Kelembapan", "Suhu", "DMI", "ONI"]])
    X, y = create_sequences(scaled, n_steps)
    if len(X) < 1:
        print(f"Skipping {province} due to insufficient sequences.")
        continue

    # Split
    split = int(len(X) * 0.8)
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    # 1-Layer LSTM
    model = Sequential()
    model.add(LSTM(30, input_shape=(n_steps, X.shape[2])))
    model.add(Dropout(0.2))
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=0.1), loss='mse')

    # Train
    early_stop = EarlyStopping(monitor='val_loss', patience=100, restore_best_weights=True)
    history = model.fit(X_train, y_train, epochs=2000, batch_size=1, verbose=0,
              callbacks=[early_stop],
              validation_data=(X_test, y_test))
    training_histories[province] = history.history

    # Predict
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    y_train_inv = inverse_scale(y_train.reshape(-1, 1), scaler)
    y_train_pred_inv = inverse_scale(y_train_pred, scaler)
    y_test_inv = inverse_scale(y_test.reshape(-1, 1), scaler)
    y_test_pred_inv = inverse_scale(y_test_pred, scaler)

    # Metrics
    train_rmse = math.sqrt(mean_squared_error(y_train_inv, y_train_pred_inv))
    train_mae = mean_absolute_error(y_train_inv, y_train_pred_inv)
    test_rmse = math.sqrt(mean_squared_error(y_test_inv, y_test_pred_inv))
    test_mae = mean_absolute_error(y_test_inv, y_test_pred_inv)
    train_r2 = r2_score(y_train_inv, y_train_pred_inv)
    test_r2 = r2_score(y_test_inv, y_test_pred_inv)

    performance_results.append({
        'Province': province,
        'Model': 'LSTM_1Layer_6Features',
        'Train RMSE': train_rmse,
        'Train MAE': train_mae,
        'Train R2': train_r2,
        'Test RMSE': test_rmse,
        'Test MAE': test_mae,
        'Test R2': test_r2
    })

# Save results
performance_df = pd.DataFrame(performance_results)
performance_df.to_csv("performance_results_1Layer.csv", index=False)
print("\nPerformance Summary:")
print(performance_df)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Processing: KEP. BANGKA BELITUNG


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 205ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 207ms/step

Processing: NUSA TENGGARA BARAT


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 205ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 216ms/step

Processing: NUSA TENGGARA TIMUR


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 216ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 196ms/step

Performance Summary:
               Province                  Model   Train RMSE    Train MAE  \
0  KEP. BANGKA BELITUNG  LSTM_1Layer_6Features   296.731073   267.482171   
1   NUSA TENGGARA BARAT  LSTM_1Layer_6Features  1202.514581  1091.575436   
2   NUSA TENGGARA TIMUR  LSTM_1Layer_6Features  2205.687615  2125.137722   

   Train R2   Test RMSE    Test MAE   Test R2  
0  0.037333  497.465531  395.594423 -0.415245  
1  0.074973  982.752527  832.315654 -8.293573  
2 -0.358054  247.235472  233.899239 -0.056237  
